# RUN_POLICY - 运行预训练策略 (生成评估视频)

评估脚本 `lerobot_eval.py` 会在模拟环境中运行策略，并生成 MP4 视频作为结果，**不会**启动 Rerun 实时预览。

> **提示**：以下所有命令已自动使用 `conda run -n lerobot` 前缀，无需手动激活环境即可直接运行。
>
> **无头服务器渲染配置**：如果在无头服务器（SSH 远程）运行 LIBERO/robosuite 环境，请先设置环境变量：
> ```bash
> export MUJOCO_GL=egl
> export PYOPENGL_PLATFORM=egl
> export EGL_DEVICE_ID=0
> ```

---

## 核心参数说明

- `--policy.path`: 预训练模型的 Hugging Face ID 或本地路径
- `--env.type`: 环境类型 (`pusht`, `aloha`, `libero`)
- `--env.task`: 环境任务名称 (如 `libero_10`, `libero_spatial`)
- `--eval.batch_size`: 并行评估的环境数量
- `--eval.n_episodes`: 总共评估的回合数
- `--policy.device`: 运行策略的设备 (`cuda` 或 `cpu`)
- `--seed`: 设置随机种子 (默认 1000)
- `--eval.use_async_envs`: (可选) 是否使用异步环境运行，默认为 `false`



> **环境说明**：本 Notebook 中所有 `python` 命令均已前置 `conda run -n lerobot`，确保在正确的环境中执行。
> 如果你是在 `lerobot` 环境中启动的 Jupyter，也可以手动去掉 `conda run -n lerobot` 前缀直接运行。


In [ ]:
# 安装 libero 依赖（运行 LIBERO 环境必需）
# CMAKE_POLICY_VERSION_MINIMUM=3.5 绕过 cmake 4.x 与 egl_probe 的兼容性问题
!CMAKE_POLICY_VERSION_MINIMUM=3.5 conda run -n lerobot pip install --no-build-isolation -e "./lerobot[libero]"


---

## 1. Diffusion Policy on PushT (2D 模拟)

最简单的入门示例，运行 Diffusion Policy 完成推方块任务。


In [ ]:
# 安装 pusht 依赖（运行 PushT 环境必需）
# 修复报错：ModuleNotFoundError: Package 'gym_pusht' required for env 'pusht' not found
!conda run -n lerobot pip install -e "./lerobot[pusht]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/diffusion_pusht \
--env.type=pusht \
--eval.batch_size=10 \
--eval.n_episodes=10 \
--policy.use_amp=false \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py

In [ ]:
# 安装 aloha 依赖（运行 Aloha 环境必需）
# 修复报错：ModuleNotFoundError: Package 'gym_aloha' required for env 'aloha' not found
!conda run -n lerobot pip install -e "./lerobot[aloha]"


## 2. ACT on Aloha (双臂模拟)

运行 Action Chunking Transformer (ACT) 完成双臂传递方块任务。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/act_aloha_sim_transfer_cube_human \
--env.type=aloha \
--env.task=AlohaTransferCube-v0 \
--eval.batch_size=1 \
--eval.n_episodes=10 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


## 3. Pi0 (VLA) on Libero (3D 模拟)

使用 Pi0 (1.4B) 视觉语言动作模型运行 LIBERO 基准测试。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_10 \
--eval.batch_size=1 \
--eval.n_episodes=10 \
--policy.device=cuda

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


### LIBERO 多品类评估

LIBERO 包含 **5 个任务品类**，共 **130 个任务**：

| 品类 | 标识符 | 任务数 | 最大步数 |
|:---|:---|:---|:---|
| LIBERO-Spatial | `libero_spatial` | 10 | 280 |
| LIBERO-Object | `libero_object` | 10 | 280 |
| LIBERO-Goal | `libero_goal` | 10 | 300 |
| LIBERO-90 | `libero_90` | 90 | 400 |
| LIBERO-Long | `libero_10` | 10 | 520 |

**评估单个品类**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=10

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


**同时评估多个品类（逗号分隔）**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial,libero_object,libero_goal,libero_10 \
--eval.batch_size=1 \
--eval.n_episodes=10

In [ ]:
# 查看评估结果（浏览器自动打开，Ctrl+C 停止服务器）
!conda run -n lerobot python scripts/view_eval.py


---

## 4. VLA 模型评估

### 4.1 支持 RTC 的模型（实时分块推理）

> **RTC (Real-Time Chunking)**：将 action chunk 生成视为 inpainting 问题，实现平滑实时控制。

| 模型 | 参数量 | 显存 | 特点 |
|:---|:---|:---|:---|
| **SmolVLA** | 450M | 4-8GB | 最轻量，消费级显卡 |
| **Pi0-FAST** | ~1.4B | 16GB+ | 离散动作 tokenizer，推理最快 |
| **Pi0** | 1.4B | 24GB+ | 基础通用模型 |
| **Pi0.5** | ~1.4B | 24GB+ | 开放世界泛化能力最强 |

---

#### 4.1.1 SmolVLA (450M)


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[smolvla]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/smolvla_libero \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
!conda run -n lerobot python scripts/smolvla_interactive.py --prompt "pick up the red block" --env libero


#### 4.2.2 X-VLA

**支持的模型+环境组合**：
| 模型 ID | 环境 |
|:---|:---|
| `lerobot/xvla-libero` | LIBERO (libero_spatial/goal/10, 93% SR) |
| `lerobot/xvla-widowx` | WidowX (BridgeData pick-and-place) |
| `lerobot/xvla-folding` | 布料折叠 (真实世界, 100% SR) |
| `lerobot/xvla-agibot-world` | AgileX 灵巧操作 (真实世界) |
| `lerobot/xvla-google-robot` | Google Robot (真实世界) |

**评估**：


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[pi]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0fast-libero \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
!conda run -n lerobot python scripts/pi0fast_interactive.py --prompt "pick up the red block" --env libero


#### 4.1.3 Pi0 (1.4B)

**支持的模型+环境组合**：
| 模型 ID | 环境 |
|:---|:---|
| `lerobot/pi0_libero_finetuned` | LIBERO (libero_spatial/object/goal/10/90) |
| `lerobot/pi0_libero` | LIBERO 基础模型 (用于微调起点) |
| `lerobot/pi0_base` | 通用基础模型 (用于微调起点) |

**评估**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi0_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 交互式 prompt 测试（单 episode，快速预览）
!conda run -n lerobot python scripts/pi0_interactive.py --prompt "pick up the red block" --task libero_spatial


#### 4.1.4 Pi0.5

**支持的模型+环境组合**：
| 模型 ID | 环境 |
|:---|:---|
| `lerobot/pi05_libero_finetuned` | LIBERO (libero_spatial/object/goal/10) |
| `lerobot/pi05_libero` | LIBERO 基础模型 (用于微调起点) |
| `lerobot/pi05_base` | 通用基础模型 (用于微调起点) |

**评估 + 交互式测试**：


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/pi05_libero_finetuned \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
!conda run -n lerobot python scripts/pi05_interactive.py --prompt "pick up the red block" --task libero_spatial


### 4.2 其他 VLA 模型

| 模型 | 参数量 | 显存 | 特点 |
|:---|:---|:---|:---|
| **Wall-X** | ~2B | 24GB+ | Qwen2.5-VL backbone |
| **X-VLA** | 0.9B+ | 16GB+ | 跨形态适应 |
| **GR00T** | 3B | 40GB+ | 人形机器人 |

---

#### 4.2.1 Wall-X (Qwen2.5-VL)

> **注意**：Wall-X 官方预训练权重来自 `x-square-robot/wall-oss-flow`，需自行微调后评估。


In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=x-square-robot/wall-oss-flow \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=1 \
--policy.device=cuda


#### 4.2.1 Wall-X (Qwen2.5-VL)

**支持的模型+环境组合**：
| 模型 ID | 环境 | 说明 |
|:---|:---|:---|
| `x-square-robot/wall-oss-flow` | 通用 | 官方预训练权重，需微调后评估 |

> **注意**：Wall-X 目前**没有** `lerobot/` 命名空间下的公开微调模型。

**评估**（基础模型，效果有限）：


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[xvla]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/xvla-libero \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=1 \
--policy.device=cuda


#### 4.2.3 GR00T N1.5

**支持的模型+环境组合**：
| 模型 ID | 环境 | 说明 |
|:---|:---|:---|
| `aractingi/bimanual-handover-groot-10k` | LIBERO | 非官方测试模型，仅用于代码通路测试 |

> **注意**：目前 Hugging Face 上**没有** `lerobot/` 命名空间下的公开 GR00T 预训练权重。

**评估**（测试模型，效果有限）：


In [ ]:
!conda run -n lerobot pip install flash-attn==2.7.4.post1 --no-build-isolation
!conda run -n lerobot pip install -e "./lerobot[groot]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=aractingi/bimanual-handover-groot-10k \
--env.type=libero \
--env.task=libero_spatial \
--eval.batch_size=1 \
--eval.n_episodes=1 \
--policy.device=cuda


#### 4.1.1 SmolVLA (450M)

**支持的模型+环境组合**：
| 模型 ID | 环境 |
|:---|:---|
| `lerobot/smolvla_libero` | LIBERO (libero_spatial/object/goal/10/90) |
| `lerobot/smolvla_metaworld` | MetaWorld (assembly-v2, pick-place-v2 等) |
| `lerobot/smolvla_robocasa` | RoboCasa (365 厨房任务) |
| `lerobot/smolvla_robotwin` | RoboTwin (50 双臂任务) |
| `lerobot/smolvla_libero_plus` | LIBERO-Plus (鲁棒性测试) |

**评估 + 交互式测试**：


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[metaworld]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/smolvla_metaworld \
--env.type=metaworld \
--env.task=assembly-v2 \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 交互式 prompt 测试（单 episode，快速预览）
!conda run --no-capture-output -n lerobot python scripts/smolvla_interactive.py --prompt "pick and place" --env metaworld


#### 4.3.2 SmolVLA on RoboCasa

RoboCasa 提供 365 个厨房操作任务，适合家庭服务机器人训练。

> **⚠️ 需要手动安装**（不是 pip extra，有循环依赖问题）：
> 参考文档：`lerobot/docs/source/robocasa.mdx`
> 
> **建议先跳过**，使用 libero 或 metaworld 测试。



#### 4.3.3 SmolVLA on RoboTwin

RoboTwin 2.0 提供 50 个双臂操作任务，基于 SAPIEN 物理引擎。

> **⚠️ 需要手动安装**（约 20 分钟，需要 SAPIEN + CUDA 12.1）：
> ```bash
> git clone https://github.com/RoboTwin-Platform/RoboTwin.git
> cd RoboTwin
> bash script/_install.sh
> bash script/_download_assets.sh
> export PYTHONPATH="${PYTHONPATH}:$(pwd)"
> ```
> 
> **建议先跳过**，使用 libero 或 metaworld 测试。


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[robocasa]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/smolvla_robocasa \
--env.type=robocasa \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 交互式 prompt 测试（单 episode，快速预览）
!conda run --no-capture-output -n lerobot python scripts/smolvla_interactive.py --prompt "open the drawer" --env robocasa


#### 4.3.2 SmolVLA on RoboTwin

RoboTwin 2.0 提供 50 个双臂操作任务，基于 SAPIEN 物理引擎。

> **⚠️ 需要手动安装 RoboTwin**（约 20 分钟）：
> ```bash
> git clone https://github.com/RoboTwin-Platform/RoboTwin.git
> cd RoboTwin
> bash script/_install.sh
> bash script/_download_assets.sh
> export PYTHONPATH="${PYTHONPATH}:$(pwd)"
> ```

**评估**：


In [ ]:
!conda run -n lerobot pip install -e "./lerobot[robotwin]"

In [ ]:
!conda run -n lerobot python lerobot/src/lerobot/scripts/lerobot_eval.py \
--policy.path=lerobot/smolvla_robotwin \
--env.type=robotwin \
--env.task=place_empty_cup \
--eval.batch_size=1 \
--eval.n_episodes=5 \
--policy.device=cuda

In [ ]:
# 交互式 prompt 测试（单 episode，快速预览）
!conda run --no-capture-output  -n lerobot python scripts/smolvla_interactive.py --prompt "pick up the object" --env robotwin


---

## 5. VLA 模型对比总结

| 模型 | 参数量 | 显存需求 | RTC | 特点 | 适用场景 |
|:---|:---|:---|:---|:---|:---|
| **SmolVLA** | 450M | 4-8GB | ✅ | 轻量、快速 | 消费级显卡、边缘计算 |
| **Pi0-FAST** | ~1.4B | 16GB+ | ✅ | 离散动作、推理快 | 需要快速推理的场景 |
| **Pi0** (π₀) | 1.4B | 24GB+ | ✅ | 基础通用模型 | 通用机器人控制 |
| **Pi0.5** | ~1.4B | 24GB+ | ✅ | 开放世界泛化 | 跨环境、未见场景 |
| **Wall-X** | ~2B | 24GB+ | ❌ | Qwen2.5-VL backbone | 跨形态泛化 |
| **X-VLA** | 0.9B+ | 16GB+ | ❌ | 跨形态适应 | 多种机器人形态适配 (软提示) |
| **GR00T** | 3B | 40GB+ | ❌ | 人形机器人 | NVIDIA 生态、人形操作 |

> **RTC (Real-Time Chunking)**：支持实时分块推理，适合需要平滑控制的场景。

---

## 6. 支持的仿真环境

| 环境 | 类型 | 任务数 | 渲染后端 | 安装命令 |
|:---|:---|:---|:---|:---|
| **PushT** | 2D 推方块 | 1 | Pygame | `pip install -e ".[pusht]"` |
| **Aloha** | 双臂操作 | 3 | Pybullet | `pip install -e ".[aloha]"` |
| **LIBERO** | 3D 操作 | 130 | MuJoCo | `pip install -e ".[libero]"` |
| **MetaWorld** | 机械臂 | 50+ | MuJoCo | `pip install -e ".[metaworld]"` |
| **RoboCasa** | 厨房场景 | 365 | MuJoCo | `pip install -e ".[robocasa]"` |
| **RoboTwin** | 双臂任务 | 50 | SAPIEN | `pip install -e ".[robotwin]"` |
| **LIBERO-Plus** | 鲁棒性测试 | 130 | MuJoCo | `pip install -e ".[libero]"` |

---

## 7. 查看任意评估结果

运行完成后，结果保存在 `outputs/eval/{日期}/{时间}_{任务名}` 目录下：

- **视频文件**：`outputs/eval/.../videos/` 包含每个评估回合的 MP4 录像
- **指标数据**：`eval_info.json` 包含成功率和奖励统计

可随时用以下命令查看任意一次评估结果（浏览器自动打开）：

```bash
# 查看最新结果
python scripts/view_eval.py

# 查看指定目录
python scripts/view_eval.py outputs/eval/2026-04-23/15-12-29_libero_pi0
```
